In [2]:
import numpy as np
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

x_train = np.load('../models/x_train.npy')
y_train = np.load('../models/y_train.npy')
x_test = np.load('../models/x_test.npy')
y_test = np.load('../models/y_test.npy')

print(f'Train :{x_train.shape} | test : {x_test.shape}')
print(f'y_train balance: {np.bincount(y_train.astype(int))}')

Train :(24000, 30) | test : (6000, 30)
y_train balance: [12000 12000]


In [4]:
print('Training Random Forest')
rf = RandomForestClassifier(n_estimators=200, max_depth=20,
                            random_state=42, n_jobs=-1)
rf.fit(x_train, y_train)
rf_acc = accuracy_score(y_test, rf.predict(x_test))
print(f'Random Forest Accuracy: {rf_acc*100:.2f}%')

with open('../models/rf_model.pkl', 'wb') as f:
    pickle.dump(rf, f)
print('Saved')

Training Random Forest
Random Forest Accuracy: 97.65%
Saved


In [6]:
print('XGBooosssttt')
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                    use_label_encoder=False, eval_metric='logloss',
                    random_state=42, n_jobs=-1)
xgb.fit(x_train, y_train)
xgb_acc = accuracy_score(y_test, xgb.predict(x_test))
print(f'XGBoost Accuracy: {xgb_acc*100:.2f}%')

with open('../models/xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb, f)

print('XGBoost model saved')

XGBooosssttt
XGBoost Accuracy: 97.97%
XGBoost model saved


In [8]:
print('LightGBM')
lgbm = LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                      random_state=42, n_jobs=-1, verbose=-1)
lgbm.fit(x_train, y_train)
lgbm_acc = accuracy_score(y_test, lgbm.predict(x_test))
print(f'LightGBm Accuracy: {lgbm_acc*100:.2f}%')

with open('../models/lgbm_model.pkl', 'wb') as f:
    pickle.dump(lgbm, f)

print('LightGBM model saved')

LightGBM
LightGBm Accuracy: 97.80%
LightGBM model saved


In [10]:
print('Traing Voting Ensemble')
ensemble = VotingClassifier(
    estimators=[('rf', rf), ('xgb' , xgb), ('lgbm', lgbm)],
    voting='soft'
)

ensemble.fit(x_train, y_train)

ens_acc = accuracy_score(y_test, ensemble.predict(x_test))

print(f'Ensemble Accuracy: {ens_acc*100:.2f} %')

with open('../models/ensemble_model.pkl','wb') as f:
    pickle.dump(ensemble, f)

print('Ensemble model saved')

Traing Voting Ensemble
Ensemble Accuracy: 97.87 %
Ensemble model saved


In [11]:
results = {
    'Random Forest': rf_acc,
    'XGBoost': xgb_acc,
    'LightGBM': lgbm_acc,
    'Ensemble': ens_acc
}

print('MODEL COMPARISON')
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f'{name:<20} {acc*100:.2f}%')

best_name = max(results, key=results.get)
best_model_map = {
    'Random Forest': rf,
    'XGBoost': xgb,
    'LightGBM': lgbm,
    'Ensemble': ensemble
}

best_model = best_model_map[best_name]

with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print(f'Best model: {best_name} ({results[best_name]*100:.2f}%)')
print('best_model.pkl saved')

MODEL COMPARISON
XGBoost              97.97%
Ensemble             97.87%
LightGBM             97.80%
Random Forest        97.65%
Best model: XGBoost (97.97%)
best_model.pkl saved
